# CAD-ViT Preprocessing Fidelity Audit

**Tujuan.** Bandingkan satu citra fundus yang sama melalui `original`, `paper_described`, `historical_v1`, dan `historical_v2_bg10`. Notebook ini tidak menjalankan training atau evaluasi test.

**Sumber dan status.** PDF CAD-ViT DOI 10.1109/ACCESS.2025.3572455, §III.A dan Gambar 1, menyebut crop border hitam, resize 512×512, dan Gaussian blur; area interpolation dibahas. Ambang crop 10, `cv2.INTER_AREA` sebagai pemanggilan spesifik, kernel blur 5×5, dan σ=1 berasal dari kode historis serta dicatat sebagai **implementation assumptions** bila paper tidak menetapkan nilainya. `paper_described` dioperasionalkan dengan konfigurasi historis V1, sehingga keluaran pikselnya **identik** dengan `historical_v1`. Ben Graham σ=10 dengan `4*image - 4*background + 128` adalah **experimental extension V2**, bukan klaim metode asli CAD-ViT.

Baca [panduan audit](../../docs/FIGURE_AUDIT.md) dan [matriks fidelitas](../../docs/PAPER_FIDELITY_MATRIX.md) sebelum menafsirkan gambar. Kemiripan tampilan dengan Gambar 1 tidak membuktikan kesamaan parameter atau pipeline penulis.


## 1. Input dan pemeriksaan split

Tentukan `CAD_VIT_AUDIT_IMAGE` sebagai path citra yang Anda pilih. Set `CAD_VIT_AUDIT_SPLIT=train` **hanya** dengan `CAD_VIT_AUDIT_MANIFEST` dan `CAD_VIT_AUDIT_DATASET_ROOT`; audit memeriksa bahwa path berada di baris `split=train`. Alternatif `CAD_VIT_AUDIT_SPLIT=audit` dipakai untuk citra audit terpisah yang dipastikan bukan test/validation. Jangan memakai gambar dari test split.

Output default: `preprocessing_audit_outputs/` di root repository, yang diabaikan Git; `CAD_VIT_AUDIT_OUTPUT` dapat menggantinya. Tidak ada path dataset pribadi di notebook ini. Gambar rusak menghasilkan error eksplisit; citra seluruhnya gelap mempertahankan `ValueError` historis; orientasi EXIF diterapkan sebelum visualisasi dan preprocessing.


In [ ]:
import os
import sys
from pathlib import Path

project_root = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / "reproduction/cadvit/preprocessing_audit.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Jalankan notebook dari dalam repository Diabetic-Rethinography")
sys.path.insert(0, str(project_root))
from reproduction.cadvit.preprocessing_audit import run_audit

image_path = os.environ.get("CAD_VIT_AUDIT_IMAGE", "").strip()
source_split = os.environ.get("CAD_VIT_AUDIT_SPLIT", "").strip().lower()
train_manifest = os.environ.get("CAD_VIT_AUDIT_MANIFEST", "").strip() or None
dataset_root = os.environ.get("CAD_VIT_AUDIT_DATASET_ROOT", "").strip() or None
output_root = Path(os.environ.get(
    "CAD_VIT_AUDIT_OUTPUT", str(project_root / "preprocessing_audit_outputs")
))
if not image_path:
    raise ValueError("Set CAD_VIT_AUDIT_IMAGE ke gambar train atau audit terpisah")
if source_split not in {"train", "audit"}:
    raise ValueError("Set CAD_VIT_AUDIT_SPLIT ke train atau audit; test dilarang")


## 2. Metode dan hasil aktual

Jalankan sel berikut hanya setelah input dan provenance split diperiksa. Keempat panel berasal dari **satu gambar yang sama**. Dua panel V1 harus identik; audit memeriksanya dengan `np.array_equal`. Grafik histogram menampilkan frekuensi intensitas 0–255 untuk tiap kanal RGB. Output PNG dan CSV disimpan di folder audit yang diabaikan Git. Sel ini sengaja belum memiliki output: tidak ada citra dataset yang dieksekusi dalam pembuatan notebook.


In [ ]:
from IPython.display import Image as NotebookImage, display

result, paths = run_audit(
    image_path=image_path,
    source_split=source_split,
    output_root=output_root,
    train_manifest=train_manifest,
    dataset_root=dataset_root,
)
print("Input SHA-256:", result["input_sha256"])
print("EXIF orientation tag:", result["exif_orientation"])
print("paper_described == historical_v1:", result["paper_v1_identical"])
print("Metrics CSV:", paths["metrics_csv"])
print("Histogram CSV:", paths["histograms_csv"])
display(NotebookImage(filename=str(paths["comparison"])))
display(NotebookImage(filename=str(paths["rgb_histograms"])))


## 3. Interpretasi dan keterbatasan

Baca nilai aktual di `metrics.csv`: brightness adalah rerata luminance `0.299R + 0.587G + 0.114B`; contrast adalah simpangan baku populasi luminance. `mean_rgb_distance_from_original` adalah norma Euclidean selisih rerata RGB pada skala 0–255. Karena crop/resize mengubah geometri, metrik rerata terhadap original adalah **deskriptif**, bukan ukuran warna piksel-sejajar. `pixel_mae_from_v1_aligned` hanya dibandingkan antara tiga keluaran yang sudah berada pada geometri 512×512 yang sama. Semua definisi dan histogram 256 bin ada di modul audit.

Amati apakah border hilang, detail lesi menjadi lebih lembut, atau V2 menimbulkan halo/saturasi. Catat observasi untuk gambar yang benar-benar dipilih; satu gambar tidak mendukung klaim peningkatan klasifikasi. Kesamaan visual dengan Gambar 1 hanya bukti tampilan, sedangkan kesamaan metode membutuhkan parameter, urutan, dan kode penulis yang belum tersedia secara lengkap. Tidak ada nilai hasil nyata yang diisi sebelum notebook dijalankan.
